# Module 2: Simple Regression Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline

## Step 1: Explore Data

In [ ]:
df = pd.read_csv("../data/combined_monthly.csv")

print("First 5 rows:")
display(df.head())

print("\nDataFrame info:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all"))

## Step 2: Split Data

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
numeric_df = df.select_dtypes(include=[np.number]).copy()

target_col = "SP500"
feature_candidates = [c for c in numeric_df.columns if c != target_col]

if not feature_candidates:
    raise ValueError("No numeric feature columns available for simple regression.")

# Pick one feature for simple regression: highest absolute correlation with SP500.
abs_corr = numeric_df[feature_candidates].corrwith(numeric_df[target_col]).abs().sort_values(ascending=False)
best_feature = abs_corr.index[0]

X = numeric_df[[best_feature]]
y = numeric_df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Selected feature: {best_feature}")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## Step 3: Develop and Fit the Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.4f}")
print(f"Coefficient for {best_feature}: {model.coef_[0]:.4f}")

## Step 4: Predict Using the Model

In [ ]:
y_pred = model.predict(X_test)

print("Predicted SP500 (first 10):")
print(np.round(y_pred[:10], 2))

## Step 5: Evaluate the Accuracy

### Plot Prediction Results and Calculate Metrics

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.7)

line_min = min(y_test.min(), y_pred.min())
line_max = max(y_test.max(), y_pred.max())
plt.plot([line_min, line_max], [line_min, line_max], "r--", linewidth=2)

plt.xlabel("Actual SP500")
plt.ylabel("Predicted SP500")
plt.title("Actual vs Predicted SP500")
plt.grid(alpha=0.2)
plt.show()

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R^2: {r2:.4f}")